In [0]:
import json
from typing import Any

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
CATALOG = "opensky_lakehouse"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.opensky_states_raw"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.flight_states"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

display(bronze_df.limit(10))

In [0]:
flight_state_schema = T.ArrayType(
    T.StructType(
        [
            T.StructField("source", T.StringType(), True),
            T.StructField("retrieved_at", T.StringType(), True),
            T.StructField("api_time", T.LongType(), True),
            T.StructField("aircraft_count", T.IntegerType(), True),

            T.StructField("icao24", T.StringType(), True),
            T.StructField("callsign", T.StringType(), True),
            T.StructField("origin_country", T.StringType(), True),
            T.StructField("time_position", T.LongType(), True),
            T.StructField("last_contact", T.LongType(), True),
            T.StructField("longitude", T.DoubleType(), True),
            T.StructField("latitude", T.DoubleType(), True),
            T.StructField("baro_altitude_m", T.DoubleType(), True),
            T.StructField("on_ground", T.BooleanType(), True),
            T.StructField("velocity_ms", T.DoubleType(), True),
            T.StructField("true_track", T.DoubleType(), True),
            T.StructField("vertical_rate", T.DoubleType(), True),
            T.StructField("sensors", T.StringType(), True),
            T.StructField("geo_altitude_m", T.DoubleType(), True),
            T.StructField("squawk", T.StringType(), True),
            T.StructField("spi", T.BooleanType(), True),
            T.StructField("position_source", T.IntegerType(), True),
        ]
    )
)

In [0]:
def get_item(values: list[Any], index: int) -> Any:
    if values is None:
        return None

    if index >= len(values):
        return None

    return values[index]


def safe_str(value: Any) -> str | None:
    if value is None:
        return None

    return str(value)


def safe_int(value: Any) -> int | None:
    if value is None:
        return None

    try:
        return int(value)
    except Exception:
        return None


def safe_float(value: Any) -> float | None:
    if value is None:
        return None

    try:
        return float(value)
    except Exception:
        return None


def safe_bool(value: Any) -> bool | None:
    if value is None:
        return None

    if isinstance(value, bool):
        return value

    if isinstance(value, str):
        value_lower = value.lower().strip()

        if value_lower == "true":
            return True

        if value_lower == "false":
            return False

    return None


def safe_json_string(value: Any) -> str | None:
    if value is None:
        return None

    try:
        return json.dumps(value)
    except Exception:
        return str(value)

In [0]:
def parse_opensky_payload(raw_json: str) -> list[dict[str, Any]]:
    if raw_json is None:
        return []

    try:
        payload = json.loads(raw_json)
    except Exception:
        return []

    source = payload.get("source")
    retrieved_at = payload.get("retrieved_at")
    api_time = safe_int(payload.get("api_time"))
    aircraft_count = safe_int(payload.get("aircraft_count"))

    states = payload.get("states") or []

    parsed_states = []

    for state in states:
        if not isinstance(state, list):
            continue

        parsed_states.append(
            {
                "source": source,
                "retrieved_at": retrieved_at,
                "api_time": api_time,
                "aircraft_count": aircraft_count,

                "icao24": safe_str(get_item(state, 0)),
                "callsign": safe_str(get_item(state, 1)),
                "origin_country": safe_str(get_item(state, 2)),
                "time_position": safe_int(get_item(state, 3)),
                "last_contact": safe_int(get_item(state, 4)),
                "longitude": safe_float(get_item(state, 5)),
                "latitude": safe_float(get_item(state, 6)),
                "baro_altitude_m": safe_float(get_item(state, 7)),
                "on_ground": safe_bool(get_item(state, 8)),
                "velocity_ms": safe_float(get_item(state, 9)),
                "true_track": safe_float(get_item(state, 10)),
                "vertical_rate": safe_float(get_item(state, 11)),
                "sensors": safe_json_string(get_item(state, 12)),
                "geo_altitude_m": safe_float(get_item(state, 13)),
                "squawk": safe_str(get_item(state, 14)),
                "spi": safe_bool(get_item(state, 15)),
                "position_source": safe_int(get_item(state, 16)),
            }
        )

    return parsed_states


parse_opensky_payload_udf = F.udf(
    parse_opensky_payload,
    flight_state_schema
)

In [0]:
parsed_df = (
    bronze_df
    .withColumn("parsed_states", parse_opensky_payload_udf(F.col("raw_json")))
)

exploded_df = (
    parsed_df
    .select(
        "source_file",
        "raw_hash",
        "bronze_ingestion_timestamp",
        F.explode(F.col("parsed_states")).alias("state")
    )
)

display(exploded_df.limit(20))

In [0]:
silver_df = (
    exploded_df
    .select(
        F.from_unixtime(F.col("state.api_time")).cast("timestamp").alias("event_time"),
        F.to_timestamp(F.col("state.retrieved_at")).alias("retrieved_at"),

        F.col("state.source").alias("source"),
        F.col("state.api_time").alias("api_time"),
        F.col("state.aircraft_count").alias("aircraft_count"),

        F.lower(F.trim(F.col("state.icao24"))).alias("icao24"),
        F.trim(F.col("state.callsign")).alias("callsign"),
        F.col("state.origin_country").alias("origin_country"),

        F.col("state.time_position").alias("time_position"),
        F.col("state.last_contact").alias("last_contact"),

        F.col("state.longitude").alias("longitude"),
        F.col("state.latitude").alias("latitude"),
        F.col("state.baro_altitude_m").alias("baro_altitude_m"),
        F.col("state.geo_altitude_m").alias("geo_altitude_m"),

        F.col("state.on_ground").alias("on_ground"),
        F.col("state.velocity_ms").alias("velocity_ms"),
        (F.col("state.velocity_ms") * F.lit(3.6)).alias("velocity_kmh"),

        F.col("state.true_track").alias("true_track"),
        F.col("state.vertical_rate").alias("vertical_rate"),
        F.col("state.sensors").alias("sensors"),
        F.col("state.squawk").alias("squawk"),
        F.col("state.spi").alias("spi"),
        F.col("state.position_source").alias("position_source"),

        F.col("source_file"),
        F.col("raw_hash"),
        F.col("bronze_ingestion_timestamp"),
        F.current_timestamp().alias("silver_ingestion_timestamp")
    )
)

In [0]:
silver_df = (
    silver_df
    .withColumn("event_date", F.to_date(F.col("event_time")))
    .withColumn("event_hour", F.hour(F.col("event_time")))
    .withColumn(
        "has_valid_coordinates",
        (
            F.col("longitude").between(-180, 180)
            & F.col("latitude").between(-90, 90)
        )
    )
    .withColumn(
        "is_valid_record",
        (
            F.col("icao24").isNotNull()
            & F.col("event_time").isNotNull()
            & F.col("has_valid_coordinates")
        )
    )
    .withColumn(
        "state_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("raw_hash"), F.lit("")),
                F.coalesce(F.col("icao24"), F.lit("")),
                F.coalesce(F.col("api_time").cast("string"), F.lit("")),
                F.coalesce(F.col("longitude").cast("string"), F.lit("")),
                F.coalesce(F.col("latitude").cast("string"), F.lit(""))
            ),
            256
        )
    )
)

display(silver_df.limit(20))

In [0]:
silver_valid_df = (
    silver_df
    .filter(F.col("is_valid_record") == True)
)

print(f"Registros Silver totales parseados: {silver_df.count()}")
print(f"Registros Silver válidos: {silver_valid_df.count()}")
print(f"Registros Silver inválidos: {silver_df.count() - silver_valid_df.count()}")

In [0]:
table_exists = spark.catalog.tableExists(SILVER_TABLE)

if table_exists:
    existing_hashes_df = (
        spark.table(SILVER_TABLE)
        .select("state_hash")
        .distinct()
    )

    silver_new_df = (
        silver_valid_df
        .join(existing_hashes_df, on="state_hash", how="left_anti")
    )
else:
    silver_new_df = silver_valid_df

print(f"Nuevos registros a insertar en Silver: {silver_new_df.count()}")

In [0]:
(
    silver_new_df
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("event_date")
    .saveAsTable(SILVER_TABLE)
)

In [0]:
spark.sql(f"""
SELECT
    event_date,
    COUNT(*) AS total_records,
    COUNT(DISTINCT icao24) AS unique_aircraft,
    COUNT(DISTINCT origin_country) AS unique_origin_countries,
    AVG(velocity_kmh) AS avg_velocity_kmh,
    AVG(baro_altitude_m) AS avg_baro_altitude_m,
    SUM(CASE WHEN on_ground = true THEN 1 ELSE 0 END) AS records_on_ground,
    SUM(CASE WHEN on_ground = false THEN 1 ELSE 0 END) AS records_in_air
FROM {SILVER_TABLE}
GROUP BY event_date
ORDER BY event_date DESC
""").display()